# Chapter 5: A/B Testing. Computing exercises

These exercises are from *Bayes, Artificial Intelligence, and Deep Learning: Foundations of Data Science* by Nick Polson and Vadim Sokolov, <https://github.com/VadimSokolov/bayes_ai_deep_learning>. The text is licensed under CC BY 4.0 and the code under the MIT License.

The code of the chapter, which several exercises refer to, is in [`code/05-ab.R`](https://github.com/VadimSokolov/bayes_ai_deep_learning/blob/main/code/05-ab.R).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VadimSokolov/bayes_ai_deep_learning/blob/main/exercises/05-ab.ipynb)

In [ ]:
# The exercises of this notebook need no package besides base R

# The exercises read files from data/: download the datasets of the book once
if (!dir.exists("data")) {
  options(timeout = max(3600, getOption("timeout")))
  download.file("https://github.com/VadimSokolov/bayes_ai_deep_learning/releases/latest/download/data.zip", "data.zip", mode = "wb")
  unzip("data.zip")
  unlink("data.zip")
}

## Exercise 5.26: A/B Testing: Banner Ad Click-Through

Use dataset from `ab_browser_test.csv`. Here is the definition of the columns:

-   `userID`: unique user ID
-   `browser`: browser which was used by userID
-   `slot`: status of the user (exp = saw modified page, control = saw
    unmodified page)
-   `n_clicks`: number of total clicks user did during as a result of
    `n_queries`
-   `n_queries`: number of queries made by `userID`, who used browser
    `browser`
-   `n_nonclk_queries`: number of queries that did not result in any
    clicks

Note, that not everyone uses a single browser, so there might be
multiple rows with the same `userID`. In this data set combination of
`userID` and `browser` is the unique row identifier.

a) Count how many users in each group. How much larger (in percent) exp
group when compared to control group

b) Using bootstrap, construct 95% confidence interval for mean and
median of number of clicks in group exp and group control. Are the
mean and median significantly different?

c) Using bootstrap, check if mean of each group has a normal
distribution. Generate $B = 1000$ bootstrap samples, calculate mean
of each and plot `qqplot`.

d) Use z-ratio for the means, to perform sis testing, with $H_0$: there
is no difference in average number of clicks between 2 groups

e) Mann-Whitney (<http://www.statmethods.net/stats/nonparametric.html>)
is another test for comparing means, that does not require normality
assumption. Use this test to check hypothesis that means are equal.

f) For each browser type and each of the 2 groups (control and exp)
count the percent of queries that did not result in any clicks. You
can do it be dividing sum of `n_nonclk_queries` by sum of
`n_queries`. Comment your on your results.

## Exercise 5.27: Statistical Power for a Two-Proportion A/B Test

A website is running an A/B test on its checkout page. The baseline (control) conversion rate is $p_1 = 0.10$, and the design team hopes the new version lifts it to $p_2 = 0.12$. Before running the experiment, they want to know how many visitors per group are needed to reliably detect a difference of this size.

a) For per-group sample size $n$ ranging from 200 to 6000, compute the power of a two-sided, two-proportion test at significance level $\alpha = 0.05$ to detect the difference between $p_1$ and $p_2$. Use the `power.prop.test` function from the `stats` package. Plot the power curve (power vs $n$).

b) Using `power.prop.test`, find the per-group sample size $n$ needed for 80% power, and separately for 90% power.

c) Set a seed and cross-check the $n$ found for 80% power with a short Monte Carlo simulation: simulate many two-proportion tests with true rates $p_1$ and $p_2$ at that $n$, and estimate the empirical power as the fraction of simulated tests that reject $H_0$ at the 5% level. Does the simulation agree with the analytical answer from `power.prop.test`?

d) Comment on how power depends on effect size and $n$: holding $n$ fixed, what happens to power as the effect size $p_2-p_1$ shrinks? Holding the effect size fixed, how much more data are needed to reach 80% power if $\alpha$ is tightened from 0.05 to 0.01?

## Exercise 5.28: Multiple Testing: Family-Wise Error and False Discovery Rate

This chapter's discussion of multiple testing showed that testing $m=1000$ hypotheses one at a time at $\alpha = 0.05$ makes at least one false positive almost certain. This exercise quantifies that inflation by simulation and compares two standard remedies: the Bonferroni correction and the Benjamini-Hochberg (BH) procedure.

a) Simulate $m$ independent tests, all true nulls, each tested at level $\alpha = 0.05$ with no adjustment for multiplicity. For $m \in \{1, 5, 10, 20, 50, 100\}$, use simulation to estimate the family-wise error rate $\text{FWER} = \mathrm{P}\left(\text{at least one rejection} \mid \text{all } m \text{ nulls true}\right)$, and compare it to the closed form $1-(1-\alpha)^m$.

b) Now let $m = 50$, with $T_0 = 40$ true nulls and $T_1 = 10$ true alternatives (real effects of size $\mu = 1.5$ on a $z$-scale). Over many replications, apply three procedures at nominal level $\alpha = 0.05$: (i) no correction, (ii) Bonferroni ($p < \alpha/m$), and (iii) Benjamini-Hochberg targeting FDR $\le \alpha$. For each procedure, report the realized FWER, the realized FDR $= \mathrm{E}\left(\frac{FD}{D}\right)$, and the power (fraction of the $T_1$ true effects detected).

c) Compare the three procedures. Which controls FWER? Which controls FDR while retaining more power? What does this imply about when to prefer Bonferroni versus Benjamini-Hochberg in practice?

## Exercise 5.29: Split-Plot ANOVA: Whole-Plot and Subplot Error Strata

This exercise makes the split-plot precision tradeoff concrete and shows why the analysis must respect the nesting. A streaming company assigns a CDN configuration (standard vs. enhanced) to each of 8 regional server clusters, 4 clusters per configuration, and within every cluster randomly splits sessions 50/50 between two recommendation algorithms. The response is a per-session engagement score. This is a split-plot layout: CDN is the hard-to-change whole-plot factor (constant within a cluster) and the algorithm is the easy-to-change subplot factor (varying within a cluster). Because sessions in the same cluster share network conditions, they are positively correlated (here an intraclass correlation near $0.15$, induced by a cluster-level random effect).

a) Simulate the design below. It has a true algorithm lift of $0.5$ engagement points and no true CDN effect; the only between-cluster signal is the random cluster effect. Fit the correct split-plot ANOVA with `aov(y ~ CDN * algo + Error(cluster))`, which separates a whole-plot (`cluster`) error stratum from a within-cluster (subplot) stratum. Report the $F$-test for CDN, for the algorithm, and for their interaction, and state which error stratum each is tested in.

b) Fit the naive model `aov(y ~ CDN * algo)` that ignores the cluster nesting and pools everything into a single residual. Compare the CDN $F$-statistic and $p$-value against the correct split-plot analysis.

c) The CDN sum of squares is identical in both analyses. Explain, in terms of the error mean square each analysis divides by, why the naive CDN $F$-statistic is inflated, and relate the inflation factor to the design-effect logic.

d) State the practical lesson for A/B tests in which a factor is randomized at a coarse level (region, store, server cluster) rather than per user.